### Importing the libraries

In [ ]:
import tensorflow as tf
import random
import numpy as np
from keras.layers import LayerNormalization, Layer, Dense, ReLU, Dropout,TextVectorization, Embedding
from keras.backend import softmax 
from tensorflow import math, matmul, reshape, shape, transpose, cast, float32

In [2]:
import keras
keras.__version__

'2.15.0'

### The Word embedding and Positional encoding layer(uses sinusoidal encoding)

In [3]:
class PositionEmbeddingFixedWeights(Layer):
    def __init__(self, seq_length, vocab_size, output_dim, **kwargs):
        super().__init__(**kwargs)
        word_embedding_matrix = self.get_position_encoding(vocab_size, output_dim)
        pos_embedding_matrix = self.get_position_encoding(seq_length, output_dim)
        self.word_embedding_layer = Embedding(input_dim=vocab_size, output_dim=output_dim,weights=[word_embedding_matrix],trainable=False)
        self.position_embedding_layer = Embedding(input_dim=seq_length, output_dim=output_dim,weights=[pos_embedding_matrix],trainable=False)

    def get_position_encoding(self, seq_len, d, n=10000):
        P = np.zeros((seq_len, d))
        for k in range(seq_len):
            for i in np.arange(int(d/2)):
                denominator = np.power(n, 2*i/d)
                P[k, 2*i] = np.sin(k/denominator)
                P[k, 2*i+1] = np.cos(k/denominator)
        return P


    def call(self, inputs):
        position_indices = tf.range(tf.shape(inputs)[-1])
        embedded_words = self.word_embedding_layer(inputs)
        embedded_indices = self.position_embedding_layer(position_indices)
        return embedded_words + embedded_indices

In [4]:
# class PositionEmbeddingFixedWeights(Layer):
#     def __init__(self, seq_length, vocab_size, output_dim, **kwargs):
#         super().__init__(**kwargs)
#         # No need to precompute positional embedding matrix for trainable embeddings
#         self.word_embedding_layer = Embedding(input_dim=vocab_size, output_dim=output_dim, trainable=True)
#         self.position_embedding_layer = Embedding(input_dim=seq_length, output_dim=output_dim, trainable=True)
#     
#     def call(self, inputs):
#         position_indices = tf.range(tf.shape(inputs)[-1])
#         embedded_words = self.word_embedding_layer(inputs)
#         embedded_indices = self.position_embedding_layer(position_indices)
#         return embedded_words + embedded_indices


### Attention mechanism

#### Single head attention

In [5]:
class DotProductAttention(Layer):
    def __init__(self,**kwargs):
        super().__init__(**kwargs)
    
    def call(self,queries,keys,values,d_k,mask = None):
        
        # Scoring the queries against the keys after transposing the latter, and scaling
        scores = matmul(queries, keys, transpose_b=True) / math.sqrt(cast(d_k, float32))
        
        # Apply mask to the attention scores
        if mask is not None:
            scores += -1e9 * mask
        
        # Computing the weights by a softmax operation
        weights = softmax(scores)
        
        # Computing the attention by a weighted sum of the value vectors
        return matmul(weights, values)
        
    

#### Multi - head Attention

In [6]:
class MultiHeadAttention(Layer):
    def __init__(self, h, d_k, d_v, d_model, **kwargs):
        super().__init__(**kwargs)
        self.attention = DotProductAttention() # Scaled dot product attention
        self.heads = h # Number of attention heads to use
        self.d_k = d_k # Dimensionality of the linearly projected queries and keys
        self.d_v = d_v # Dimensionality of the linearly projected values
        self.d_model = d_model # Dimensionality of the model
        self.W_q = Dense(d_k) # Learned projection matrix for the queries
        self.W_k = Dense(d_k) # Learned projection matrix for the keys
        self.W_v = Dense(d_v) # Learned projection matrix for the values
        self.W_o = Dense(d_model) # Learned projection matrix for the multi-head output
    def reshape_tensor(self, x, heads, flag):
        if flag:
            # Tensor shape after reshaping and transposing:
            # (batch_size, heads, seq_length, -1)
            x = reshape(x, shape=(shape(x)[0], shape(x)[1], heads, -1))
            x = transpose(x, perm=(0, 2, 1, 3))
        else:
            # Reverting the reshaping and transposing operations:
            # (batch_size, seq_length, d_k)
            x = transpose(x, perm=(0, 2, 1, 3))
            x = reshape(x, shape=(shape(x)[0], shape(x)[1], self.d_k))
        return x
    
    def call(self, queries, keys, values, mask=None):
        # Rearrange the queries to be able to compute all heads in parallel
        q_reshaped = self.reshape_tensor(self.W_q(queries), self.heads, True)
        # Resulting tensor shape: (batch_size, heads, input_seq_length, -1)
        # Rearrange the keys to be able to compute all heads in parallel
        k_reshaped = self.reshape_tensor(self.W_k(keys), self.heads, True)
        # Resulting tensor shape: (batch_size, heads, input_seq_length, -1)
        # Rearrange the values to be able to compute all heads in parallel
        v_reshaped = self.reshape_tensor(self.W_v(values), self.heads, True)
        # Resulting tensor shape: (batch_size, heads, input_seq_length, -1)
        # Compute the multi-head attention output using the reshaped queries,
        # keys, and values
        o_reshaped = self.attention(q_reshaped, k_reshaped, v_reshaped, self.d_k, mask)
        # Resulting tensor shape: (batch_size, heads, input_seq_length, -1)
        # Rearrange back the output into concatenated form
        output = self.reshape_tensor(o_reshaped, self.heads, False)
        # Resulting tensor shape: (batch_size, input_seq_length, d_v)
        # Apply one final linear projection to the output to generate the multi-head
        # attention. Resulting tensor shape: (batch_size, input_seq_length, d_model)
        return self.W_o(output)

### Normalization 

In [7]:
# Implementing the Add & Norm Layer
class AddNormalization(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layer_norm = LayerNormalization() # Layer normalization layer
    def call(self, x, sublayer_x):
        # The sublayer input and output need to be of the same shape to be summed
        add = x + sublayer_x
        # Apply layer normalization to the sum
        return self.layer_norm(add)

### Feed forward layer

In [8]:
# Implementing the Feed-Forward Layer
class FeedForward(Layer):
    def __init__(self, d_ff, d_model, **kwargs):
        super().__init__(**kwargs)
        self.fully_connected1 = Dense(d_ff) # First fully connected layer
        self.fully_connected2 = Dense(d_model) # Second fully connected layer
        self.activation = ReLU() # ReLU activation layer
        
    def call(self, x):
        # The input is passed into the two fully-connected layers, with a ReLU in between
        x_fc1 = self.fully_connected1(x)
        return self.fully_connected2(self.activation(x_fc1))

### Encoder stack

#### A single Encoder layer

In [9]:
# Implementing the Encoder Layer
class EncoderLayer(Layer):
    def __init__(self, h, d_k, d_v, d_model, d_ff, rate, **kwargs):
        super().__init__(**kwargs)
        self.multihead_attention = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout1 = Dropout(rate)
        self.add_norm1 = AddNormalization()
        self.feed_forward = FeedForward(d_ff, d_model)
        self.dropout2 = Dropout(rate)
        self.add_norm2 = AddNormalization()
    
    def call(self, x, padding_mask, training):
        # Multi-head attention layer
        multihead_output = self.multihead_attention(x, x, x, padding_mask)
        
        # Expected output shape = (batch_size, sequence_length, d_model)
        # Add in a dropout layer
        multihead_output = self.dropout1(multihead_output, training=training)
        # Followed by an Add & Norm layer
        addnorm_output = self.add_norm1(x, multihead_output)
        # Expected output shape = (batch_size, sequence_length, d_model)
        # Followed by a fully connected layer
        feedforward_output = self.feed_forward(addnorm_output)
        # Expected output shape = (batch_size, sequence_length, d_model)
        # Add in another dropout layer
        feedforward_output = self.dropout2(feedforward_output, training=training)
        # Followed by another Add & Norm layer
        return self.add_norm2(addnorm_output, feedforward_output)

#### The stacked encoder

In [10]:
# Implementing the Encoder
class Encoder(Layer):
    def __init__(self, vocab_size, sequence_length, h, d_k, d_v, d_model, d_ff, n, rate,
    **kwargs):
        super().__init__(**kwargs)
        self.pos_encoding = PositionEmbeddingFixedWeights(sequence_length, vocab_size,
        d_model)
        self.dropout = Dropout(rate)
        self.encoder_layer = [EncoderLayer(h, d_k, d_v, d_model, d_ff, rate) for _ in range(n)]
    
    def call(self, input_sentence, padding_mask, training):
        # Generate the positional encoding
        pos_encoding_output = self.pos_encoding(input_sentence)
        # Expected output shape = (batch_size, sequence_length, d_model)
        # Add in a dropout layer
        x = self.dropout(pos_encoding_output, training=training)
        # Pass on the positional encoded values to each encoder layer
        for i, layer in enumerate(self.encoder_layer):
            x = layer(x, padding_mask, training)
        return x

### Main code

In [11]:
h = 8 # Number of self-attention heads
d_k = 64 # Dimensionality of the linearly projected queries and keys
d_v = 64 # Dimensionality of the linearly projected values
d_ff = 2048 # Dimensionality of the inner fully connected layer
d_model = 512 # Dimensionality of the model sub-layers' outputs
n = 6 # Number of layers in the encoder stack
batch_size = 64 # Batch size from the training process
dropout_rate = 0.1 # Frequency of dropping the input units in the dropout layers

In [12]:
enc_vocab_size = 20 # Vocabulary size for the encoder
input_seq_length = 5 # Maximum length of the input sequence
input_seq = np.random.rand(batch_size, input_seq_length)

In [13]:
input_seq

array([[4.59318724e-01, 5.10520512e-01, 6.74566029e-01, 4.99532820e-01,
        3.65339978e-02],
       [2.46098470e-01, 3.80051147e-01, 3.67154024e-01, 9.81663474e-01,
        4.22931575e-01],
       [5.58991511e-01, 5.26204909e-01, 2.30210607e-01, 3.44238523e-01,
        2.54167489e-01],
       [4.66008959e-01, 2.84190329e-01, 9.39782234e-01, 8.10999011e-02,
        7.87270253e-01],
       [7.01111805e-01, 5.93613003e-02, 5.71366298e-01, 1.82051170e-01,
        9.36746815e-01],
       [4.40641264e-01, 5.50875028e-01, 3.31161713e-01, 4.65695448e-01,
        5.56763061e-02],
       [3.62543616e-01, 6.37968219e-01, 4.27914739e-02, 1.57444044e-02,
        6.48546081e-01],
       [5.05971797e-01, 9.58714395e-01, 4.96874118e-01, 6.35386218e-01,
        4.61640730e-01],
       [6.16410143e-02, 7.09928245e-01, 1.06676239e-01, 6.67400258e-01,
        8.02908782e-01],
       [5.21357861e-01, 1.60767178e-01, 9.03749712e-02, 4.38586097e-01,
        5.10696521e-02],
       [9.26247751e-01, 6.1896

In [14]:
input_seq.shape

(64, 5)

In [15]:
encoder = Encoder(enc_vocab_size, input_seq_length, h, d_k, d_v, d_model, d_ff, n,
dropout_rate)
res = encoder(input_seq, None, True)

2025-01-22 23:16:36.428670: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2025-01-22 23:16:36.428713: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-01-22 23:16:36.428758: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-01-22 23:16:36.429049: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-01-22 23:16:36.429222: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Running this code produces an output of shape (batch size, sequence length, model
dimensionality). 

In [16]:
print(res)

tf.Tensor(
[[[-1.72230914e-01 -2.06433129e+00  2.79341549e-01 ... -7.98856974e-01
    9.36670415e-03 -6.35484993e-01]
  [-4.49666642e-02  1.64519221e-01 -2.77306825e-01 ... -1.01001695e-01
    8.80277038e-01  1.09365009e-01]
  [ 4.18045893e-02 -2.57720757e+00  1.16926268e-01 ... -1.42145589e-01
    2.47727334e-01 -4.73866373e-01]
  [-3.42766158e-02 -2.67487764e+00 -2.96458751e-02 ... -4.19305593e-01
    4.80154872e-01 -6.40347004e-01]
  [-7.85275042e-01 -2.44248772e+00 -2.05324233e-01 ... -5.89971364e-01
   -1.55438930e-02 -6.15740418e-01]]

 [[-1.50287867e-01 -1.94274235e+00 -7.54909277e-01 ... -2.77807593e-01
   -5.89744329e-01 -5.03030598e-01]
  [-1.23615324e+00 -2.57315612e+00 -1.76540062e-01 ... -5.02316892e-01
    2.44760424e-01 -2.58940279e-01]
  [-3.66153836e-01 -1.80830613e-01 -7.00888857e-02 ...  1.75584018e-01
   -1.42031506e-01 -6.40081614e-02]
  [ 1.97499439e-01 -2.42636585e+00 -6.84370339e-01 ...  3.54053527e-02
   -6.12812579e-01 -2.03772798e-01]
  [-4.69447821e-01 -2.84

In [17]:
res.shape

TensorShape([64, 5, 512])